In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("setup successfully 🎉")

setup successfully 🎉


## Dataset description (Olympics dataset)

For this analysis, we use the **Olympics dataset** from Kaggle:  
**Source:** https://www.kaggle.com/datasets/harshvgh/olympics

This dataset contains historical Olympic records and is commonly split into two files:

1. **athlete_events.csv**  
    - Event-level data for athletes across Olympic Games  
    - Typical fields include athlete details (`Name`, `Sex`, `Age`, `Height`, `Weight`), team/country (`Team`, `NOC`), competition info (`Games`, `Year`, `Season`, `City`, `Sport`, `Event`), and result (`Medal`)

2. **noc_regions.csv**  
    - Mapping table from **NOC code** to country/region name  
    - Used to enrich and standardize country-level analysis

Together, these files support analysis of participation trends, gender distribution, sport/event popularity, and medal outcomes over time.

In [3]:
athletes = pd.read_csv("Data/athlete_events.csv")
athletes.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


In [4]:
regions = pd.read_csv("Data/noc_regions.csv")
regions.head()

,NOC,region,notes
0,AFG,Afghanistan,NaN
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,NaN
3,ALG,Algeria,NaN
4,AND,Andorra,NaN


---

## 🎯 Project Roadmap

### 1. **Setup & Load** 📦
   - Import libraries, load datasets

### 2. **Explore & Clean** 🔍
   - Check shape, dtypes, missing values, duplicates

### 3. **Merge & Enrich** 🔗
   - Join athletes + regions for country analysis

### 4. **EDA & Visualize** 📊
   - Participation trends over time
   - Gender distribution
   - Top countries by medals
   - Age/Height/Weight distributions

### 5. **Deep Dives** 🏅
   - Sport-wise analysis
   - Summer vs Winter comparison
   - Medal prediction insights

### 6. **Key Takeaways** 💡
   - Summarize findings & conclusions


In [5]:
# Stage 2
athletes.info()
athletes.describe()
print('\n')
print(athletes.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  object 
 2   Sex     271116 non-null  object 
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  object 
 7   NOC     271116 non-null  object 
 8   Games   271116 non-null  object 
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  object 
 11  City    271116 non-null  object 
 12  Sport   271116 non-null  object 
 13  Event   271116 non-null  object 
 14  Medal   39783 non-null   object 
dtypes: float64(3), int64(2), object(10)
memory usage: 31.0+ MB


(271116, 15)


> We'll only analyze summer olympics ✅ not winter olympics ❌

In [6]:
df = athletes[athletes['Season'] == "Summer"]
df.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
26,8,"Cornelia ""Cor"" Aalten (-Strannood)",F,18.0,168.0,NaN,Netherlands,NED,1932 Summer,1932,Summer,Los Angeles,Athletics,Athletics Women's 100 metres,NaN


> now let's merge both the datsets

In [7]:
df = df.merge(regions,on="NOC", how="left")
df.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,region,notes
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN,China,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN,China,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN,Denmark,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold,Denmark,NaN
4,8,"Cornelia ""Cor"" Aalten (-Strannood)",F,18.0,168.0,NaN,Netherlands,NED,1932 Summer,1932,Summer,Los Angeles,Athletics,Athletics Women's 100 metres,NaN,Netherlands,NaN


> let's handle missing and duplicate values

In [8]:
df.isnull().sum()
# honestly we are ignoring it right now

ID             0
Name           0
Sex            0
Age         9189
Height     51857
Weight     53854
Team           0
NOC            0
Games          0
Year           0
Season         0
City           0
Sport          0
Event          0
Medal     188464
region       370
notes     218151
dtype: int64

In [9]:
print(df.duplicated().sum())
df.drop_duplicates(inplace=True)
df.duplicated().sum()


1385


np.int64(0)

In [10]:
df.Medal.value_counts()

Medal
Gold      11456
Bronze    11409
Silver    11212
Name: count, dtype: int64

In [11]:
encod = pd.get_dummies(df.Medal).astype(int)
encod

,Bronze,Gold,Silver
0,0,0,0
1,0,0,0
2,0,0,0
3,0,1,0
4,0,0,0
...,...,...,...
222547,0,0,0
222548,0,0,0
222549,0,0,0
222550,0,0,0


In [12]:
df = pd.concat([df, encod], axis=1)
print('shape of df dataset is :', df.shape)
df.head()


shape of df dataset is : (221167, 20)


,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,region,notes,Bronze,Gold,Silver
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN,China,NaN,0,0,0
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN,China,NaN,0,0,0
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN,Denmark,NaN,0,0,0
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold,Denmark,NaN,0,1,0
4,8,"Cornelia ""Cor"" Aalten (-Strannood)",F,18.0,168.0,NaN,Netherlands,NED,1932 Summer,1932,Summer,Los Angeles,Athletics,Athletics Women's 100 metres,NaN,Netherlands,NaN,0,0,0


In [13]:
df.groupby('NOC')[['Gold', 'Silver', 'Bronze']].sum().sort_values('Gold', ascending=False).head(10)

,Gold,Silver,Bronze
NOC,,,
USA,2472,1333,1197
URS,832,635,596
GBR,635,729,620
GER,592,538,649
ITA,518,474,454
FRA,463,567,587
HUN,432,328,363
SWE,354,396,358
AUS,342,452,510


## ⚠ Why This Medal Tally Doesn't Match the Official Olympic Tally

The grouped result above does **not** match the official Olympic medal tally because of the following reasons:

###  Team vs Individual Counting
- The dataset may count **each athlete medal entry separately**.
- Official tallies count **one medal per event**, not per athlete.



In [14]:
# df.Sport.value_counts()
df2 = df.drop_duplicates(subset=['Team', 'NOC','Games','Year','Season','City','Sport', 'Event', 'Medal'])
# Team	NOC	Games	Year	Season	City	Sport	Event
medal_tally = df2.groupby('region')[['Gold', 'Silver', 'Bronze']].sum().sort_values('Gold', ascending=False)

# medal tally is df for medal tally of each country in each olympics
medal_tally

,Gold,Silver,Bronze
region,,,
USA,1035,802,708
Russia,592,498,487
Germany,444,457,491
UK,278,317,300
France,234,256,287
...,...,...,...
Vanuatu,0,0,0
"Virgin Islands, British",0,0,0
"Virgin Islands, US",0,1,0


> now we'll create a list of 
- country
- years

In [15]:
country_list =  ['Overall'] + medal_tally.index.sort_values().tolist()
# country = medal_tally.index.tolist()
country_list

['Overall',
 'Afghanistan',
 'Albania',
 'Algeria',
 'American Samoa',
 'Andorra',
 'Angola',
 'Antigua',
 'Argentina',
 'Armenia',
 'Aruba',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bahamas',
 'Bahrain',
 'Bangladesh',
 'Barbados',
 'Belarus',
 'Belgium',
 'Belize',
 'Benin',
 'Bermuda',
 'Bhutan',
 'Boliva',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'Brunei',
 'Bulgaria',
 'Burkina Faso',
 'Burundi',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Cape Verde',
 'Cayman Islands',
 'Central African Republic',
 'Chad',
 'Chile',
 'China',
 'Colombia',
 'Comoros',
 'Cook Islands',
 'Costa Rica',
 'Croatia',
 'Cuba',
 'Curacao',
 'Cyprus',
 'Czech Republic',
 'Democratic Republic of the Congo',
 'Denmark',
 'Djibouti',
 'Dominica',
 'Dominican Republic',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'Equatorial Guinea',
 'Eritrea',
 'Estonia',
 'Ethiopia',
 'Fiji',
 'Finland',
 'France',
 'Gabon',
 'Gambia',
 'Georgia',
 'Germany',
 'Ghana',
 'Greece',
 'Grenada',
 'Guam',
 'Guatemala',
 'Gui

In [18]:
year_list = ['Overall'] + df.Year.sort_values().unique().tolist()
year_list

['Overall',
 1896,
 1900,
 1904,
 1906,
 1908,
 1912,
 1920,
 1924,
 1928,
 1932,
 1936,
 1948,
 1952,
 1956,
 1960,
 1964,
 1968,
 1972,
 1976,
 1980,
 1984,
 1988,
 1992,
 1996,
 2000,
 2004,
 2008,
 2012,
 2016]

In [23]:
def fetch_Medal_tally(country,year):
    if country == 'Overall' and year == 'Overall':
        return medal_tally
    elif country == 'Overall' and year != 'Overall':
        df_year = df2[df2['Year'] == year]
        return df_year
    elif country != 'Overall' and year == 'Overall':
        df_country = df2[df2['region'] == country]
        return df_country
    else:
        df_country_year = df2[(df2['region'] == country) & (df2['Year'] == year)]
        return df_country_year
    
fetch_Medal_tally('Saudi Arabia','Overall')[['Gold','Silver','Bronze']].sum()

Gold      0
Silver    1
Bronze    2
dtype: int64

In [28]:
def fetch_Medal_tally(country,year):
    if country == 'Overall' and year == 'Overall':
        return medal_tally
    elif country == 'Overall' and year != 'Overall':

        # here we want to return
        df_year = df2[df2['Year'] == year]
        medal_tally_year = df_year.groupby('region')[['Gold', 'Silver', 'Bronze']].sum().sort_values('Gold', ascending=False)
        return medal_tally_year
    elif country != 'Overall' and year == 'Overall':
        df_country = df2[df2['region'] == country]

        medal_tally_country = df_country.groupby('Year')[['Gold', 'Silver', 'Bronze']].sum().sort_values('Year', ascending=True)
        # medal_tally_country = df_country.groupby('region')[['Gold', 'Silver', 'Bronze']].sum().sort_values('Gold', ascending=False)
        return medal_tally_country
    else:
        df_country_year = df2[(df2['region'] == country) & (df2['Year'] == year)]
        medal_tally_country_year = df_country_year.groupby('region')[['Gold', 'Silver', 'Bronze']].sum().sort_values('Gold', ascending=False)
        return medal_tally_country_year

In [ ]:
fetch_Medal_tally('USA','Overall')

,Gold,Silver,Bronze
Year,,,
1972,0,0,0
1976,0,0,0
1984,0,0,0
1988,0,0,0
1992,0,0,0
1996,0,0,0
2000,0,1,1
2004,0,0,0
2008,0,0,0
